In [ ]:
# Code from binns_DDP.py, just setting up all datasets
import csv
import functools
import math
import sys
import time
import random
import warnings
import subprocess
import argparse
from mlp import mlp_wrapper
#from pe_gcn_model import GridCellSpatialRelationEncoder
from sklearn.model_selection import KFold

# sys.path.append('C:/Users/hx293/Research_Data/BINN/')
# sys.path.append('/glade/u/home/haodixu/BINN')
# sys.path.append(r'/User/homes/ftao/Projects/BINNS/src_binns')
sys.path.append('/glade/work/haodixu/BINN')

# Set HDF5_DISABLE_VERSION_CHECK to suppress version mismatch error
import os
os.environ['HDF5_DISABLE_VERSION_CHECK'] = '2'
import psutil
import gc
from datetime import datetime, timedelta
import pandas as pd
from pandas import DataFrame as df
import numpy as np
from scipy.interpolate import pchip_interpolate
import os
import torch
from torch import nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import random_split, DataLoader
# from torch.utils.tensorboard import SummaryWriter
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.data.distributed import DistributedSampler
import multiprocessing
from multiprocessing import Process
from scipy.io import loadmat
import netCDF4 as ncread 
import mat73
from matplotlib import pyplot as plt
from collections import OrderedDict

#####################################
# Import Different Versions of CLM5 #
#####################################

# from fun_matrix_clm5 import fun_model_simu
from fun_matrix_clm5_vectorized import fun_model_simu, fun_model_prediction
import visualization_utils
from fun_matrix_clm5_vectorized_bulk_converge import fun_bulk_simu

################################################
# Command-line arguments
################################################
args = argparse.Namespace()
args.seed = 0
args.n_datapoints = 500
args.embed_dim = 10
args.vertical_mixing = 'original'
args.whether_resume = 0
args.cross_val_idx = 0

In [ ]:
def set_seeds(seed):
	"""
	Attempts to set all random seeds to improve reproducibility.
	"""
	random.seed(seed)
	np.random.seed(seed) # set the random seed of numpy
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = True

# Set seeds for reproducibility
set_seeds(args.seed)

# Keep track of when the job started
job_begin_time = time.time()


#################################################################################
# ATTENTION: There is a lot of code that is not inside any function,
# which sets up datasets. On CPU (start_method=fork), this gets run
# ONCE, and the forked processes have access to all variables created.
# On GPU (start_method=spawn), this gets run once initially, and AGAIN
# for each process. It would be nice to factor this out eventually.
################################################################################# 

################################################
# Data Directories (CHANGE THIS!!!)
################################################
# data_dir_input = '/Users/phoenix/Google_Drive/Tsinghua_Luo/Projects/DATAHUB/ENSEMBLE/INPUT_DATA/'
# data_dir_output = '/Users/phoenix/Google_Drive/Tsinghua_Luo/Projects/DATAHUB/BINNS/OUTPUT_DATA/'
# data_dir_input = 'C:/Users/hx293/Research_Data/BINN/ENSEMBLE/INPUT_DATA/'
# data_dir_output = 'C:/Users/hx293/Unsync_Data/BINN_output/'
# server path
# job_submit_path = '/glade/u/home/haodixu/BINN/PBS_Submit/Bulk_Converge/'
# data_dir_input = '/glade/u/home/haodixu/BINN/ENSEMBLE/INPUT_DATA/'
# data_dir_output = '/glade/work/haodixu/BINN/BINNS/OUTPUT_DATA/'
# data_dir_input = '/glade/u/home/haodixu/BINN/ENSEMBLE/INPUT_DATA/'
# data_dir_output = '/glade/work/joshuaf/BINNS/OUTPUT_DATA/'
# job_submit_path = '/glade/work/joshuaf/BINNS/src_binns/resume_jobs/'
# os.makedirs(data_dir_output, exist_ok=True)
# os.makedirs(job_submit_path, exist_ok=True)
data_dir_input = '/mnt/beegfs/bulk/mirror/jyf6/datasets/BINNS/INPUT_DATA/'
data_dir_output = '/mnt/beegfs/bulk/mirror/jyf6/datasets/BINNS/OUTPUT_DATA/'
job_submit_path = '/mnt/beegfs/bulk/mirror/jyf6/datasets/BINNS/src_binns/resume_jobs/'
os.makedirs(job_submit_path, exist_ok=True)

################################################
# Setup datasets
################################################
cesm2_case_name = 'sasu_f05_g16_checked_step4'
start_year = 661
end_year = 680

time_domain = 'whole_time' # 'whole_time', 'before_1985', 'after_1985', 'random_half_1', 'random_half_2'
model_name = 'cesm2_clm5_cen_vr_v2'

start_id = 1
end_id = 5000
is_resubmit = 0

# constants
month_num = 12 
soil_cpool_num = 7
soil_decom_num = 20

#-------------------------------
# wosis data
#-------------------------------
# load wosis data

# The site information for each SOC profile. 
# Names for each column are "profile_id" "country_id" "country_name" "lon" "lat" "layer_num" “date”. 
nc_data_middle = ncread.Dataset(data_dir_input + 'wosis_2019_snap_shot/soc_profile_wosis_2019_snapshot_hugelius_mishra.nc') # wosis profile info
wosis_profile_info = nc_data_middle['soc_profile_info'][:].data.transpose()
nc_data_middle.close()

# The full dataset which contains SOC content information at each layer
# layer_info: "profile_id, date, upper_depth, lower_depth, node_depth, soc_layer_weight, soc_stock, bulk_denstiy, is_pedo"
nc_data_middle = ncread.Dataset(data_dir_input + 'wosis_2019_snap_shot/soc_data_integrate_wosis_2019_snapshot_hugelius_mishra.nc') # wosis SOC info
wosis_soc_info = nc_data_middle['data_soc_integrate'][:].data.transpose()
nc_data_middle.close()

#-------------------------------
# PRODA Predicted Parameters
#-------------------------------
# load PRODA predicted parameters

# The site information for each parameter prediction.
# Get the profile id for predicted parameters
# data from nn_site_loc_full_cesm2_clm5_cen_vr_v2_whole_time_exp_pc_cesm2_23_cross_valid_0_1.csv to 9
for i in range(1, 10):
	# contains one column of profile id
	nn_site_loc_temp = pd.read_csv(data_dir_input + 'PRODA_Results/nn_site_loc_full_cesm2_clm5_cen_vr_v2_whole_time_exp_pc_cesm2_23_cross_valid_0_' + str(i) + '.csv', header=None)
	# contains the predicted parameters (21) for each profile
	nn_site_para_temp = pd.read_csv(data_dir_input + 'PRODA_Results/nn_para_result_full_cesm2_clm5_cen_vr_v2_whole_time_exp_pc_cesm2_23_cross_valid_0_' + str(i) + '.csv', header=None)
	# create a dataframe to store the profile id and the parameters
	if i == 1:
		# initialize the dataframe
		PRODA_para = pd.DataFrame(nn_site_loc_temp)
		# rename the column
		PRODA_para.columns = ['profile_id']
		# add the parameters
		PRODA_para = pd.concat([PRODA_para, nn_site_para_temp], axis = 1)
	else:
		# add the parameters
		PRODA_para = pd.concat([PRODA_para, nn_site_para_temp], axis = 1)
# end
# Get the mean value for each parameter for each profile
for i in range(1,22):
	PRODA_para['mean_' + str(i)] = PRODA_para.iloc[:, i:21*10:21].mean(axis = 1)
# end
# Drop the original columns
PRODA_para = PRODA_para.drop(PRODA_para.columns[1:21*9], axis = 1)
# print the head of the dataframe
print("PRODA parameters")
print(PRODA_para.head())

#-------------------------------
# CLM5 constants
#-------------------------------
# Parameter names
if args.vertical_mixing == 'original':
	para_names = ['diffus', 'cryo', 'q10', 'efolding', 'taucwd', 'taul1', 'taul2', 'tau4s1', 'tau4s2', 'tau4s3', 'fl1s1', 'fl2s1', 'fl3s2', 'fs1s2', 'fs1s3', 'fs2s1', 'fs2s3', 'fs3s1', 'fcwdl2', 'w-scaling', 'beta']
else:
	# If using the simpler vertical mixing parameterization, replace diffus/cryo with slope/intercept.
	para_names = ['slope', 'intercept', 'q10', 'efolding', 'taucwd', 'taul1', 'taul2', 'tau4s1', 'tau4s2', 'tau4s3', 'fl1s1', 'fl2s1', 'fl3s2', 'fs1s2', 'fs1s3', 'fs2s1', 'fs2s3', 'fs3s1', 'fcwdl2', 'w-scaling', 'beta']
	if args.vertical_mixing == 'simple_two_intercepts':
		para_names.append('intercept_leach')

# Parameters index for retrieval test
para_index = np.arange(0, len(para_names))  # If choosing all parameters
# para_index = [0, 2, 3, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20]

# Soil depths info
# width between two interfaces
dz = np.array([2.000000000000000E-002, 4.000000000000000E-002, 6.000000000000000E-002, \
8.000000000000000E-002, 0.120000000000000, 0.160000000000000, \
0.200000000000000, 0.240000000000000, 0.280000000000000, \
0.320000000000000, 0.360000000000000, 0.400000000000000, \
0.440000000000000, 0.540000000000000, 0.640000000000000, \
0.740000000000000, 0.840000000000000, 0.940000000000000, \
1.04000000000000, 1.14000000000000, 2.39000000000000, \
4.67553390593274, 7.63519052838329, 11.1400000000000, \
15.1154248593737])

# depth of the interface
zisoi = np.array([2.000000000000000E-002, 6.000000000000000E-002, \
0.120000000000000, 0.200000000000000, 0.320000000000000, \
0.480000000000000, 0.680000000000000, 0.920000000000000, \
1.20000000000000, 1.52000000000000, 1.88000000000000, \
2.28000000000000, 2.72000000000000, 3.26000000000000, \
3.90000000000000, 4.64000000000000, 5.48000000000000, \
6.42000000000000, 7.46000000000000, 8.60000000000000, \
10.9900000000000, 15.6655339059327, 23.3007244343160, \
34.4407244343160, 49.5561492936897])

zisoi_0 = 0

# depth of the node
zsoi = np.array([1.000000000000000E-002, 4.000000000000000E-002, 9.000000000000000E-002, \
0.160000000000000, 0.260000000000000, 0.400000000000000, \
0.580000000000000, 0.800000000000000, 1.06000000000000, \
1.36000000000000, 1.70000000000000, 2.08000000000000, \
2.50000000000000, 2.99000000000000, 3.58000000000000, \
4.27000000000000, 5.06000000000000, 5.95000000000000, \
6.94000000000000, 8.03000000000000, 9.79500000000000, \
13.3277669529664, 19.4831291701244, 28.8707244343160, \
41.9984368640029])

# depth between two node
dz_node = zsoi - np.append(np.array([0]), zsoi[:-1], axis = 0)


# cesm2 resolution
cesm2_resolution_lat = 180/384
cesm2_resolution_lon = 360/576
lon_grid = np.arange((-180 + cesm2_resolution_lon/2), 180, cesm2_resolution_lon)
lat_grid = np.arange((90 - cesm2_resolution_lat/2), -90, -cesm2_resolution_lat)

# load cesm2 input
var_name_list = ['nbedrock', 'ALTMAX', 'ALTMAX_LASTYEAR', 'CELLSAND', 'NPP', \
	'SOILPSI', 'TSOI', \
	'W_SCALAR', 'T_SCALAR', 'O_SCALAR', 'FPI_vr', \
	'LITR1_INPUT_ACC_VECTOR', 'LITR2_INPUT_ACC_VECTOR', 'LITR3_INPUT_ACC_VECTOR', 'CWD_INPUT_ACC_VECTOR', \
	'TOTSOMC']

var_name_list_rename =  ['cesm2_simu_nbedrock', 'cesm2_simu_altmax', 'cesm2_simu_altmax_last_year', 'cesm2_simu_cellsand', 'cesm2_simu_npp', \
	'cesm2_simu_soil_water_potnetial', 'cesm2_simu_soil_temperature', \
	'cesm2_simu_w_scalar', 'cesm2_simu_t_scalar', 'cesm2_simu_o_scalar', 'cesm2_simu_n_scalar', \
	'cesm2_simu_input_vector_litter1', 'cesm2_simu_input_vector_litter2', 'cesm2_simu_input_vector_litter3', 'cesm2_simu_input_vector_cwd', \
	'cesm2_simu_soc_stock']

for ivar in np.arange(0, len(var_name_list)):
	# load simulation from CESM2
	var_record_monthly_mean = mat73.loadmat(data_dir_input + 'cesm2_simu/spinup_ss/' + cesm2_case_name + '_cesm2_ss_4da_' + str(start_year) + '_' + str(end_year) + '_' + var_name_list[ivar] + '.mat')
	var_record_monthly_mean = var_record_monthly_mean['var_record_monthly_mean']
	exec(var_name_list_rename[ivar] + ' = var_record_monthly_mean')
# end

for ilayer in np.arange(0, soil_decom_num):
	cesm2_simu_input_vector_litter1[:, :, ilayer, :] = cesm2_simu_input_vector_litter1[:, :, ilayer, :]*dz[ilayer]
	cesm2_simu_input_vector_litter2[:, :, ilayer, :] = cesm2_simu_input_vector_litter2[:, :, ilayer, :]*dz[ilayer]
	cesm2_simu_input_vector_litter3[:, :, ilayer, :] = cesm2_simu_input_vector_litter3[:, :, ilayer, :]*dz[ilayer]
	cesm2_simu_input_vector_cwd[:, :, ilayer, :] = cesm2_simu_input_vector_cwd[:, :, ilayer, :]*dz[ilayer]
#end

cesm2_simu_input_sum_litter1 = np.sum(cesm2_simu_input_vector_litter1, axis = 2)
cesm2_simu_input_sum_litter2 = np.sum(cesm2_simu_input_vector_litter2, axis = 2)
cesm2_simu_input_sum_litter3 = np.sum(cesm2_simu_input_vector_litter3, axis = 2)
cesm2_simu_input_sum_cwd = np.sum(cesm2_simu_input_vector_cwd, axis = 2)

del cesm2_simu_input_vector_litter1, cesm2_simu_input_vector_litter2, cesm2_simu_input_vector_litter3, cesm2_simu_input_vector_cwd


############################################
# Select subset of observations (profiles) #
############################################
# Not used currently
# representative points 
sample_profile_id = loadmat(data_dir_input + 'wosis_2019_snap_shot/wosis_2019_snapshot_hugelius_mishra_representative_profiles.mat')
sample_profile_id = sample_profile_id['sample_profile_id']
# convert the number to be starting from 0 in python world
sample_profile_id = sample_profile_id - 1

### Use 2000 profiles for testing ###
# profile_collection = np.reshape(sample_profile_id[:, 0:20], [2000, 1])

# if use the whole dataset
# profile_collection = np.arange(0, 20000)
# if select 
# profile_collection = np.arange(0, wosis_profile_info.shape[0])
# profile_collection = np.reshape(profile_collection, [profile_collection.shape[0], 1])	

# Choose the profile id with lat and lon within the range of the United States
profile_collection = np.where(
    (wosis_profile_info[:, 2] == 156) & 
    (wosis_profile_info[:, 3] >= -124.763068) & 
    (wosis_profile_info[:, 3] <= -66.949895) & 
    (wosis_profile_info[:, 4] >= 24.5) & 
    (wosis_profile_info[:, 4] <= 49.384358)
)[0]

################################
# If use same dataset as PRODA #
################################
# load mat file
para_gr = loadmat(data_dir_input + 'wosis_2019_snap_shot/cesm2_clm5_cen_vr_v2_para_gr.mat')
stat_r2 = loadmat(data_dir_input + 'wosis_2019_snap_shot/cesm2_clm5_cen_vr_v2_stat_r2.mat')
eligible_profile = loadmat(data_dir_input + 'wosis_2019_snap_shot/eligible_profile_loc_0_cesm2_clm5_cen_vr_v2_whole_time.mat')
para_gr = para_gr['para_gr']
stat_r2 = stat_r2['stat_r2']
eligible_profile = eligible_profile['eligible_loc_0']
# convert the number to be starting from 0 in python world
eligible_profile = eligible_profile - 1
# calculate average value per row in para_gr, and choose those profiles with average value less than 1.05
# calculate average value per row in stat_r2, and choose those profiles with average value larger than 0
# choose profile that listed in eligible_profile
PRODA_collection = np.where((np.mean(para_gr, axis = 1) < 1.05) & 
							(np.mean(stat_r2, axis = 1) > 0) & 
							(np.isin(np.arange(0, wosis_profile_info.shape[0]), eligible_profile) == True) & 
							# Also in the column profile_id of the dataframe PRODA_para
							(np.isin(np.arange(0, wosis_profile_info.shape[0]), PRODA_para['profile_id']) == True)
							)[0]
# Choose overlap between profile_collection and PRODA_collection
profile_collection = np.intersect1d(profile_collection, PRODA_collection)

if args.n_datapoints != -1:
	# Choose random subset of profiles for testing.
	# TODO: maybe create a separate data seed, to separate randomness in data selection from
	# randomness in algorithm/model initialization
	rng = np.random.default_rng(seed=args.seed)
	profile_collection = rng.choice(profile_collection, args.n_datapoints, replace=False)

profile_collection = np.reshape(profile_collection, [profile_collection.shape[0], 1])
profile_range = np.arange(0, len(profile_collection))
print('number of profiles: ', len(profile_collection))
print(datetime.now(), '------------all input data loaded------------')

In [ ]:
#---------------------------------------------------
# wrap up soc data for NN
#---------------------------------------------------
obs_soc_matrix = np.ones([len(profile_collection), 200])*np.nan  # Each row is a profile. Each non-nan column is an SOC observation
obs_depth_matrix = np.ones([len(profile_collection), 200])*np.nan  # Each row is a profile. Each column represents the depth of the corresponding SOC observation in "obs_soc_matrix"
obs_upper_depth_matrix = np.ones([len(profile_collection), 200])*np.nan  # Each row is a profile. Each column represents the upper depth of the corresponding SOC observation in "obs_soc_matrix"
obs_lower_depth_matrix = np.ones([len(profile_collection), 200])*np.nan  # Each row is a profile. Each column represents the lower depth of the corresponding SOC observation in "obs_soc_matrix"
obs_lon_lat_loc = np.ones([len(profile_collection), 2])*np.nan

model_force_input_vector_cwd = np.ones([len(profile_collection), month_num])*np.nan
model_force_input_vector_litter1 = np.ones([len(profile_collection), month_num])*np.nan
model_force_input_vector_litter2 = np.ones([len(profile_collection), month_num])*np.nan
model_force_input_vector_litter3 = np.ones([len(profile_collection), month_num])*np.nan

model_force_altmax_lastyear_profile = np.ones([len(profile_collection), month_num])*np.nan
model_force_altmax_current_profile = np.ones([len(profile_collection), month_num])*np.nan
model_force_nbedrock = np.ones([len(profile_collection), month_num])*np.nan

model_force_xio = np.ones([len(profile_collection), soil_decom_num, month_num])*np.nan
model_force_xin = np.ones([len(profile_collection), soil_decom_num, month_num])*np.nan

model_force_sand_vector = np.ones([len(profile_collection), soil_decom_num, month_num])*np.nan

model_force_soil_temp_profile = np.ones([len(profile_collection), soil_decom_num, month_num])*np.nan
model_force_soil_water_profile = np.ones([len(profile_collection), soil_decom_num, month_num])*np.nan

# record the sum of recorded layers for all profiles
layer_num_record = 0

for iprofile_hat in profile_range:
	# profile num
	iprofile = profile_collection[iprofile_hat]
	# profile id
	profile_id = wosis_profile_info[iprofile, 0]
	# find currently using profile
	loc_profile = np.where(wosis_soc_info[:, 0] == profile_id)[0]
	# find the lon and lat info of soil profile
	lon_profile = wosis_profile_info[iprofile, 3]
	lat_profile = wosis_profile_info[iprofile, 4]
	
	lat_loc = np.where(abs(lat_profile - lat_grid) == min(abs(lat_profile - lat_grid)))[0][0]
	lon_loc = np.where(abs(lon_profile - lon_grid) == min(abs(lon_profile - lon_grid)))[0][0]
	
	# info of the node depth of profile  
	wosis_layer_depth = wosis_soc_info[loc_profile, 4]
	# observed C info (gC/m3)
	wosis_layer_obs = wosis_soc_info[loc_profile, 6]
	# check how many layers are recorded
	layer_num_record = layer_num_record + len(wosis_layer_obs)
	# observced upper depth of each layer
	wosis_layer_upper_depth = wosis_soc_info[loc_profile, 2]
	# observced lower depth of each layer
	wosis_layer_lower_depth = wosis_soc_info[loc_profile, 3]
	# exclude nan values
	valid_soc_loc = np.where((np.isnan(wosis_layer_obs) == False) & (np.isnan(wosis_layer_depth) == False) & (np.isnan(wosis_layer_upper_depth) == False) & (np.isnan(wosis_layer_lower_depth) == False))
	# valid layer number
	num_layers = len(valid_soc_loc[0])
	
	if num_layers > 0:
		wosis_layer_depth = wosis_layer_depth[valid_soc_loc]/100 # convert unit from cm to m
		wosis_layer_obs = wosis_layer_obs[valid_soc_loc]
		wosis_layer_upper_depth = wosis_layer_upper_depth[valid_soc_loc]/100
		wosis_layer_lower_depth = wosis_layer_lower_depth[valid_soc_loc]/100
		
		obs_depth_matrix[iprofile_hat, 0:num_layers] = wosis_layer_depth
		obs_soc_matrix[iprofile_hat, 0:num_layers] = wosis_layer_obs
		obs_upper_depth_matrix[iprofile_hat, 0:num_layers] = wosis_layer_upper_depth
		obs_lower_depth_matrix[iprofile_hat, 0:num_layers] = wosis_layer_lower_depth
	# end if num_layers > 0:


	# NOT USED CURRENTLY: Interpolate SOC observations to the 20 layers we predict for.
	# (Instead, we interpolate our PREDICTIONS to the observation depths)
	# if num_layers > 1:
	# 	wosis_layer_depth = wosis_layer_depth[valid_soc_loc]/100 # convert unit from cm to m
	# 	wosis_layer_obs = wosis_layer_obs[valid_soc_loc]
	# 	
	# 	wosis_layer_depth = wosis_layer_depth + (np.random.rand(num_layers)-0.5)*10**(-7)
	# 	sort_index = np.argsort(wosis_layer_depth)
	# 	wosis_layer_depth = wosis_layer_depth[sort_index]
	# 	wosis_layer_obs = wosis_layer_obs[sort_index]
	# 	
	# 	interp_soc = pchip_interpolate(wosis_layer_depth, wosis_layer_obs, zsoi)
	# 	interp_soc[interp_soc <= 0] = np.nan
	# 	interp_start_loc = np.where(abs(wosis_layer_depth[0] - zsoi) == min(abs(wosis_layer_depth[0] - zsoi)))[0]
	# 	interp_end_loc = np.where(abs(wosis_layer_depth[-1] - zsoi) == min(abs(wosis_layer_depth[-1] - zsoi)))[0]
	# 	if (interp_start_loc < 19) & (interp_end_loc < 19):
	# 		# print('multilayer profile: ', iprofile_hat)
	# 		obs_soc_matrix[iprofile_hat, interp_start_loc[0]:interp_end_loc[0]] = interp_soc[interp_start_loc[0]:interp_end_loc[0]]
	# elif num_layers == 1:
	# 	wosis_layer_depth = wosis_layer_depth[valid_soc_loc]/100 # convert unit from cm to m
	# 	wosis_layer_obs = wosis_layer_obs[valid_soc_loc]
	# 	
	# 	closest_loc = np.where(abs(wosis_layer_depth[0] - zsoi) == min(abs(wosis_layer_depth[0] - zsoi)))[0]
	# 	obs_soc_matrix[iprofile_hat, closest_loc] = wosis_layer_obs[0]
	# elif num_layers == 0:
	# 	print('invalid profile: ', iprofile_hat)
	# # end if num_layers > 3:

	obs_lon_lat_loc[iprofile_hat, :] = [lon_loc, lat_loc]
	
	# input vector
	model_force_input_vector_cwd[iprofile_hat, :] = cesm2_simu_input_sum_cwd[lat_loc, lon_loc, :]
	model_force_input_vector_litter1[iprofile_hat, :] = cesm2_simu_input_sum_litter1[lat_loc, lon_loc, :]
	model_force_input_vector_litter2[iprofile_hat, :] = cesm2_simu_input_sum_litter2[lat_loc, lon_loc, :]
	model_force_input_vector_litter3[iprofile_hat, :] = cesm2_simu_input_sum_litter3[lat_loc, lon_loc, :]
	# altmax current and last year
	model_force_altmax_lastyear_profile[iprofile_hat, :] = cesm2_simu_altmax_last_year[lat_loc, lon_loc, :]
	model_force_altmax_current_profile[iprofile_hat, :] = cesm2_simu_altmax[lat_loc, lon_loc, :]
	# nbedrock
	model_force_nbedrock[iprofile_hat, :] = cesm2_simu_nbedrock[lat_loc, lon_loc, :]
	# oxygen scalar
	model_force_xio[iprofile_hat, :, :] = cesm2_simu_o_scalar[lat_loc, lon_loc, 0:soil_decom_num, :]
	# nitrogen scalar
	model_force_xin[iprofile_hat, :, :] = cesm2_simu_n_scalar[lat_loc, lon_loc, 0:soil_decom_num, :]
	# sand content
	model_force_sand_vector[iprofile_hat, :, :] = cesm2_simu_cellsand[lat_loc, lon_loc, 0:soil_decom_num, :]
	# soil temperature and water potential
	model_force_soil_temp_profile[iprofile_hat, :, :] = cesm2_simu_soil_temperature[lat_loc, lon_loc, 0:soil_decom_num, :]
	model_force_soil_water_profile[iprofile_hat, :, :] = cesm2_simu_w_scalar[lat_loc, lon_loc, 0:soil_decom_num, :]
# end

# check the overall number of layers in the profile
print("Number of layers in profile: " + str(layer_num_record))
print(datetime.now(), '------------soc data prepared------------')

########################################################
# neural network (BINNS)
########################################################
nn_split_ratio = 0.1
test_split_ratio = 0.1

#---------------------------------------------------
# env info
#---------------------------------------------------
# environmental info of soil profiles
env_info_names = ['ProfileNum', 'ProfileID', 'LayerNum', 'Lon', 'Lat', 'Date', \
'Rmean', 'Rmax', 'Rmin', \
'ESA_Land_Cover', \
'ET', \
'IGBP', 'Climate', 'Soil_Type', 'NPPmean', 'NPPmax', 'NPPmin', \
'Veg_Cover', \
'BIO1', 'BIO2', 'BIO3', 'BIO4', 'BIO5', 'BIO6', 'BIO7', 'BIO8', 'BIO9', 'BIO10', 'BIO11', 'BIO12', 'BIO13', 'BIO14', 'BIO15', 'BIO16', 'BIO17', 'BIO18', 'BIO19', \
'Abs_Depth_to_Bedrock', \
'Bulk_Density_0cm', 'Bulk_Density_30cm', 'Bulk_Density_100cm',\
'CEC_0cm', 'CEC_30cm', 'CEC_100cm', \
'Clay_Content_0cm', 'Clay_Content_30cm', 'Clay_Content_100cm', \
'Coarse_Fragments_v_0cm', 'Coarse_Fragments_v_30cm', 'Coarse_Fragments_v_100cm', \
'Depth_Bedrock_R', \
'Garde_Acid', \
'Occurrence_R_Horizon', \
'pH_Water_0cm', 'pH_Water_30cm', 'pH_Water_100cm', \
'Sand_Content_0cm', 'Sand_Content_30cm', 'Sand_Content_100cm', \
'Silt_Content_0cm', 'Silt_Content_30cm', 'Silt_Content_100cm', \
'SWC_v_Wilting_Point_0cm', 'SWC_v_Wilting_Point_30cm', 'SWC_v_Wilting_Point_100cm', \
'Texture_USDA_0cm', 'Texture_USDA_30cm', 'Texture_USDA_100cm', \
'USDA_Suborder', \
'WRB_Subgroup', \
'Drought', \
'Elevation', \
'Max_Depth', \
'Koppen_Climate_2018', \
'cesm2_npp', 'cesm2_npp_std', \
'cesm2_gpp', 'cesm2_gpp_std', \
'cesm2_vegc', \
'nbedrock', \
'R_Squared']

# variables used in training the NN
var4nn = ['Lon', 'Lat', \
'ESA_Land_Cover', \
# 'IGBP', \
# 'Climate', \
# 'Soil_Type', \
# 'NPPmean', 'NPPmax', 'NPPmin', \
# 'Veg_Cover', \
'BIO1', 'BIO2', 'BIO3', 'BIO4', 'BIO5', 'BIO6', 'BIO7', 'BIO8', 'BIO9', 'BIO10', 'BIO11', 'BIO12', 'BIO13', 'BIO14', 'BIO15', 'BIO16', 'BIO17', 'BIO18', 'BIO19', \
'Abs_Depth_to_Bedrock', \
'Bulk_Density_0cm', 'Bulk_Density_30cm', 'Bulk_Density_100cm',\
'CEC_0cm', 'CEC_30cm', 'CEC_100cm', \
'Clay_Content_0cm', 'Clay_Content_30cm', 'Clay_Content_100cm', \
'Coarse_Fragments_v_0cm', 'Coarse_Fragments_v_30cm', 'Coarse_Fragments_v_100cm', \
# 'Depth_Bedrock_R', \
'Garde_Acid', \
'Occurrence_R_Horizon', \
'pH_Water_0cm', 'pH_Water_30cm', 'pH_Water_100cm', \
'Sand_Content_0cm', 'Sand_Content_30cm', 'Sand_Content_100cm', \
'Silt_Content_0cm', 'Silt_Content_30cm', 'Silt_Content_100cm', \
'SWC_v_Wilting_Point_0cm', 'SWC_v_Wilting_Point_30cm', 'SWC_v_Wilting_Point_100cm', \
'Texture_USDA_0cm', 'Texture_USDA_30cm', 'Texture_USDA_100cm', \
'USDA_Suborder', \
'WRB_Subgroup', \
# 'Drought', \
'Elevation', \
# 'Max_Depth', \
'Koppen_Climate_2018', \
'cesm2_npp', 'cesm2_npp_std', \
# 'cesm2_gpp', 'cesm2_gpp_std', \
'cesm2_vegc', \
'nbedrock']


env_info = loadmat(data_dir_input + 'wosis_2019_snap_shot/wosis_2019_snapshot_hugelius_mishra_env_info.mat')
env_info = env_info['EnvInfo']
original_lons = env_info[:, 3].copy()
original_lats = env_info[:, 4].copy()

# Min/max for each feature
col_max_min = loadmat(data_dir_input + 'wosis_2019_snap_shot/world_grid_envinfo_present_cesm2_clm5_cen_vr_v2_whole_time_col_max_min.mat')
col_max_min = col_max_min['col_max_min']


################################################
# Categorical variables                        #
################################################
# List of categorical variables. Inner lists group categorical
# variables that share the same categories. For example, the category IDs
# in Texture_USDA_0cm, Texture_USDA_30cm have the same semantic meaning,
# so they share an embedding space.
categorical_vars = [['ESA_Land_Cover'], ['Texture_USDA_0cm', 'Texture_USDA_30cm', 'Texture_USDA_100cm'], 
					['USDA_Suborder'], ['WRB_Subgroup'], ['Koppen_Climate_2018']]  # Variables inside a sub-list share the same categories
categorical_vars_flattened = [item for sublist in categorical_vars for item in sublist]

# Don't want to transform categorical variables, so set max/min to nan
for group in categorical_vars:
	for var in group:
		idx = env_info_names.index(var)
		# print("Var {} Nans {}".format(var, np.count_nonzero(np.isnan(env_info[:, idx]))))
		col_max_min[idx, :] = np.nan


#####################################################################
# Transform covariates to [0, 1] range based on precomputed min/max #
#####################################################################
# warnings.filterwarnings("error")
for ivar in np.arange(3, len(col_max_min[:, 0])):
	if np.isnan(col_max_min[ivar, :]).any():
		pass
	else:
		env_info[:, ivar] = (env_info[:, ivar] - col_max_min[ivar, 0])/(col_max_min[ivar, 1] - col_max_min[ivar, 0])
		env_info[(env_info[:, ivar] > 1), ivar] = 1
		env_info[(env_info[:, ivar] < 0), ivar] = 0
	# except:
	# 	print('error in variable: ', ivar)
# warnings.resetwarnings()

# env_info_scaled = loadmat(data_dir_input + 'wosis_2019_snap_shot/wosis_2019_snapshot_hugelius_mishra_env_info_' + model_name  + '_' + time_domain + '_maxmin_scaled.mat')
# env_info_scaled = df(env_info_scaled['profile_env_info'])
# env_info = env_info_scaled

env_info = df(env_info)
env_info.columns = env_info_names
env_info["original_lon"] = original_lons
env_info["original_lat"] = original_lats

# # @joshuafan added temporarily
# env_info.index = env_info.ProfileNum
# print("Env info old shape", env_info.shape)
# print("Env info", env_info.head())
# print(profile_collection[0:5, 0])


# For each categorical variable, record the number of classes (categories)
var_idx_to_num_classes = dict()  # Column index to number of classes
for group in categorical_vars:
	n_categories = int(np.nanmax(env_info[group]) + 1)
	for var in group:
		idx = var4nn.index(var)
		var_idx_to_num_classes[idx] = n_categories



#---------------------------------------------------
# training data
#---------------------------------------------------
current_data_x = np.ones((len(profile_collection), len(var4nn), 12, 13))*np.nan

# ATTENTION: env_info is indexed starting from 0. The use of loc below means that 
# profile_collection is being interpreted as ZERO-BASED indices.
# However, the Profile_Num column starts at 1. Check this?
current_data_x[:, 0:len(var4nn), 0, 0] = np.array(env_info.loc[profile_collection[:, 0], var4nn])
current_data_x[:, 0:12, 0, 1] = model_force_input_vector_cwd
current_data_x[:, 0:12, 0, 2] = model_force_input_vector_litter1
current_data_x[:, 0:12, 0, 3] = model_force_input_vector_litter2
current_data_x[:, 0:12, 0, 4] = model_force_input_vector_litter3
current_data_x[:, 0:12, 0, 5] = model_force_altmax_lastyear_profile
current_data_x[:, 0:12, 0, 6] = model_force_altmax_current_profile
current_data_x[:, 0:12, 0, 7] = model_force_nbedrock

current_data_x[:, 0:20, 0:12, 8] = model_force_xio
current_data_x[:, 0:20, 0:12, 9] = model_force_xin
current_data_x[:, 0:20, 0:12, 10] = model_force_sand_vector
current_data_x[:, 0:20, 0:12, 11] = model_force_soil_temp_profile
current_data_x[:, 0:20, 0:12, 12] = model_force_soil_water_profile


current_data_y = obs_soc_matrix
current_data_z = obs_depth_matrix

lons = np.array(env_info.loc[profile_collection[:, 0], "original_lon"])
lats = np.array(env_info.loc[profile_collection[:, 0], "original_lat"])

# # VISUALIZATION: Plot map of each environmental covariate.
# for col_idx, col_name in enumerate(var4nn):
# 	envir_var_values = current_data_x[:, col_idx, 0, 0]
# 	categorical = (col_name in categorical_vars_flattened)
# 	visualization_utils.plot_observations_world_map(lons, lats, envir_var_values, PLOT_DIR, col_name, categorical=categorical)

# # VISUALIZATION: Plot map of SOC observation labels within each layer. If a profile has multiple observations 
# # in a layer, pick the first one
# layer_top = 0
# for layer_idx in range(len(zisoi)):
# 	layer_bottom = zisoi[layer_idx]
# 	this_layer_y = np.ones((current_data_y.shape[0])) * np.nan

# 	# Loop through all profiles
# 	for j in range(current_data_y.shape[0]):
# 		# Get depth of each SOC observation
# 		depths = current_data_z[j]

# 		# Select SOC observations whose depth falls within the current layer
# 		this_layer_this_profile_y = current_data_y[j, (~np.isnan(depths)) & (depths >= layer_top) & (depths < layer_bottom)]
# 		if len(this_layer_this_profile_y) > 1:
# 			continue
# 			print("Oddly enough this profile had more than 2 observations in the same soil layer")
# 			print("Layer", layer_top, "to", layer_bottom)
# 			print("Observation depths", depths)
# 		elif len(this_layer_this_profile_y) == 0:
# 			continue
# 		else:
# 			this_layer_y[j] = this_layer_this_profile_y[0]
# 	layer_name = "Layer {} ({:.2f}-{:.2f} m)".format(layer_idx, layer_top, layer_bottom)
# 	col_name = "soc_layer{}_{:.2f}-{:.2f}m".format(layer_idx, layer_top, layer_bottom)
# 	visualization_utils.plot_observations_world_map(lons, lats, this_layer_y, PLOT_DIR, col_name)
# 	layer_top = layer_bottom


nan_loc = np.nanmean(current_data_y, axis = 1) + \
			np.sum(current_data_x[:, 0:len(var4nn), 0, 0], axis = 1) + \
			np.sum(model_force_input_vector_cwd, axis = 1) + \
			np.sum(model_force_input_vector_litter1, axis = 1) + \
			np.sum(model_force_input_vector_litter2, axis = 1) + \
			np.sum(model_force_input_vector_litter3, axis = 1) + \
			np.sum(model_force_altmax_lastyear_profile, axis = 1) + \
			np.sum(model_force_altmax_current_profile, axis = 1) + \
			np.sum(model_force_nbedrock, axis = 1) + \
			np.sum(model_force_xio, axis = (1, 2)) + \
			np.sum(model_force_xin, axis = (1, 2)) + \
			np.sum(model_force_sand_vector, axis = (1, 2)) + \
			np.sum(model_force_soil_temp_profile, axis = (1, 2)) + \
			np.sum(model_force_soil_water_profile, axis = (1, 2))

valid_profile_loc = np.where(np.isnan(nan_loc) == False)[0] ### Why change the shape from 26915 to 26934??? ###

current_data_y = current_data_y[valid_profile_loc, :]
current_data_z = current_data_z[valid_profile_loc, :]
current_data_x = current_data_x[valid_profile_loc, :, :, :]
current_data_profile_id = profile_collection[valid_profile_loc, 0]
obs_upper_depth_matrix = obs_upper_depth_matrix[valid_profile_loc, :]
obs_lower_depth_matrix = obs_lower_depth_matrix[valid_profile_loc, :]
# env_info = env_info.loc[valid_profile_loc, :]
print("Shape of current data x", current_data_x.shape)
print("Shape of current data y", current_data_y.shape)
print("Shape of current data z", current_data_z.shape)
print("Shape of obs upper depth matrix", obs_upper_depth_matrix.shape)
print("Shape of obs lower depth matrix", obs_lower_depth_matrix.shape)
print("Shape of env info", env_info.shape)
# env_info = env_info.loc[valid_profile_loc, :]


# Select PRODA parameters so that the Profile_IDs match the current data
PRODA_para = PRODA_para.loc[PRODA_para['profile_id'].isin(current_data_profile_id)]
PRODA_para = PRODA_para.sort_values(by='profile_id')                 
print("Shape of PRODA para", PRODA_para.shape)


###############################################################
# Load checkpoint if resuming a previous run.
# We do this outside the main function, since the checkpoint
# stores the train/val/test split for setting up the datasets.
##################################################################
# If PREVIOUS_JOB_ID environment variable set, overwrite the
# commandline arg.
if 'PREVIOUS_JOB_ID' in os.environ:
	args.previous_job_id = os.environ.get('PREVIOUS_JOB_ID')
	print("Overrode previous_job_id. Now", args.previous_job_id)

# Load checkpoint if resuming
if args.whether_resume == 1:
	# Load Checkpoint
	checkpoint_path = data_dir_output + 'neural_network/' + args.previous_job_id + '/checkpoint_' + args.previous_job_id + '.pt'
	checkpoint_main = torch.load(checkpoint_path)

	# Delete the job submit file if it exists
	try:
		os.remove(job_submit_path + 'Resume' + args.previous_job_id + '.submit')
	except OSError:
		pass


################################################################
# Data splitting
################################################################
# k-fold cross validation
k_folds = 10

# Train, validation, test split
if args.whether_resume == 0:
	if args.cross_val_idx == 0:
		if test_split_ratio == 0:
			train_loc = np.random.choice(np.arange(0, len(current_data_x[:, 0])), size = round((1-nn_split_ratio)*len(current_data_x[:, 0])), replace = False)
			val_loc = np.setdiff1d(np.arange(0, len(current_data_x[:, 0])), train_loc)

			train_y = torch.tensor(current_data_y[train_loc, :], dtype = torch.float32)
			val_y = torch.tensor(current_data_y[val_loc, :], dtype = torch.float32)

			train_z = torch.tensor(current_data_z[train_loc, :], dtype = torch.float32)
			val_z = torch.tensor(current_data_z[val_loc, :], dtype = torch.float32)

			train_x = torch.tensor(current_data_x[train_loc, :, :, :], dtype = torch.float32)
			# train_x = train_x.requires_grad_(True)
			val_x = torch.tensor(current_data_x[val_loc, :, :, :], dtype = torch.float32)
			# val_x = val_x.requires_grad_(True)

			train_profile_id = torch.tensor(current_data_profile_id[train_loc], dtype = torch.long)
			val_profile_id = torch.tensor(current_data_profile_id[val_loc], dtype = torch.long)
		else:
			# Determine the number of training samples based on the ratios
			train_loc = np.random.choice(np.arange(0, len(current_data_x[:, 0])), size=round((1 - nn_split_ratio - test_split_ratio) * len(current_data_x[:, 0])), replace=False)
			# The remaining data after removing the training samples
			remaining_loc = np.setdiff1d(np.arange(0, len(current_data_x[:, 0])), train_loc)
			# Split the remaining data into validation and test sets
			num_val_samples = round(nn_split_ratio / (nn_split_ratio + test_split_ratio) * len(remaining_loc))
			val_loc = np.random.choice(remaining_loc, size=num_val_samples, replace=False)
			test_loc = np.setdiff1d(remaining_loc, val_loc)

			train_y = torch.tensor(current_data_y[train_loc, :], dtype=torch.float32)
			val_y = torch.tensor(current_data_y[val_loc, :], dtype=torch.float32)
			test_y = torch.tensor(current_data_y[test_loc, :], dtype=torch.float32)

			train_z = torch.tensor(current_data_z[train_loc, :], dtype=torch.float32)
			val_z = torch.tensor(current_data_z[val_loc, :], dtype=torch.float32)
			test_z = torch.tensor(current_data_z[test_loc, :], dtype=torch.float32)

			train_x = torch.tensor(current_data_x[train_loc, :, :, :], dtype=torch.float32)
			# train_x = train_x.requires_grad_(True)
			val_x = torch.tensor(current_data_x[val_loc, :, :, :], dtype=torch.float32)
			# val_x = val_x.requires_grad_(True)
			test_x = torch.tensor(current_data_x[test_loc, :, :, :], dtype=torch.float32)
			# test_x = test_x.requires_grad_(True)

			train_profile_id = torch.tensor(current_data_profile_id[train_loc], dtype=torch.long)
			val_profile_id = torch.tensor(current_data_profile_id[val_loc], dtype=torch.long)
			test_profile_id = torch.tensor(current_data_profile_id[test_loc], dtype=torch.long)

			print("Shape of train data", train_x.shape)
			print("Shape of val data", val_x.shape)
			print("Shape of test data", test_x.shape)
	else:
		# Split the data into k-folds (defined previously)
		# Assign the test dataset based on the cross-validation index
		# Randomly split the remaining data into training and validation sets
		kf = KFold(n_splits=k_folds, shuffle=True, random_state=args.seed)
		fold_indices = list(kf.split(np.arange(len(current_data_x[:, 0]))))
		test_loc = fold_indices[args.cross_val_idx - 1][1]
		train_val_idx = fold_indices[args.cross_val_idx - 1][0]

		train_loc = np.random.choice(train_val_idx, size=round((1 - nn_split_ratio - test_split_ratio)/(1 - test_split_ratio) * len(train_val_idx)), replace=False)
		val_loc = np.setdiff1d(train_val_idx, train_loc)

		train_y = torch.tensor(current_data_y[train_loc, :], dtype=torch.float32)
		val_y = torch.tensor(current_data_y[val_loc, :], dtype=torch.float32)
		test_y = torch.tensor(current_data_y[test_loc, :], dtype=torch.float32)

		train_z = torch.tensor(current_data_z[train_loc, :], dtype=torch.float32)
		val_z = torch.tensor(current_data_z[val_loc, :], dtype=torch.float32)
		test_z = torch.tensor(current_data_z[test_loc, :], dtype=torch.float32)

		train_x = torch.tensor(current_data_x[train_loc, :, :, :], dtype=torch.float32)
		# train_x = train_x.requires_grad_(True)
		val_x = torch.tensor(current_data_x[val_loc, :, :, :], dtype=torch.float32)
		# val_x = val_x.requires_grad_(True)
		test_x = torch.tensor(current_data_x[test_loc, :, :, :], dtype=torch.float32)
		# test_x = test_x.requires_grad_(True)

		train_profile_id = torch.tensor(current_data_profile_id[train_loc], dtype=torch.long)
		val_profile_id = torch.tensor(current_data_profile_id[val_loc], dtype=torch.long)
		test_profile_id = torch.tensor(current_data_profile_id[test_loc], dtype=torch.long)

		print("Shape of train data", train_x.shape)
		print("Shape of val data", val_x.shape)
		print("Shape of test data", test_x.shape)


else:
	# load train, val, and test indices
	train_loc = checkpoint_main['train_indices']
	val_loc = checkpoint_main['val_indices']
	test_loc = checkpoint_main['test_indices']
	# split the data
	train_y = torch.tensor(current_data_y[train_loc, :], dtype=torch.float32)
	val_y = torch.tensor(current_data_y[val_loc, :], dtype=torch.float32)
	test_y = torch.tensor(current_data_y[test_loc, :], dtype=torch.float32)

	train_z = torch.tensor(current_data_z[train_loc, :], dtype=torch.float32)
	val_z = torch.tensor(current_data_z[val_loc, :], dtype=torch.float32)
	test_z = torch.tensor(current_data_z[test_loc, :], dtype=torch.float32)

	train_x = torch.tensor(current_data_x[train_loc, :, :, :], dtype=torch.float32)
	val_x = torch.tensor(current_data_x[val_loc, :, :, :], dtype=torch.float32)
	test_x = torch.tensor(current_data_x[test_loc, :, :, :], dtype=torch.float32)

	train_profile_id = torch.tensor(current_data_profile_id[train_loc], dtype=torch.long)
	val_profile_id = torch.tensor(current_data_profile_id[val_loc], dtype=torch.long)
	test_profile_id = torch.tensor(current_data_profile_id[test_loc], dtype=torch.long)


print(datetime.now(), '------------nn data prepared------------')

## Single site parameters

In [ ]:
# Choose a random site's SOC observations. We want to plot the "parameter loss landscape" -
# as we perturb parameters, how much does the loss (prediction error) change.
# TODO Check location of site, make sure in temperate region, find site with more observations
# TODO Convert to NSE
n_obs = torch.sum(~torch.isnan(train_z), dim=1)
highest_obs, indices = torch.topk(n_obs, k=10)
EXAMPLE_IDX = indices[1]
example_y = train_y[EXAMPLE_IDX:EXAMPLE_IDX+1]  # SOC observations
example_x = train_x[EXAMPLE_IDX:EXAMPLE_IDX+1]  # Forcing/environmental covariates
example_z = train_z[EXAMPLE_IDX:EXAMPLE_IDX+1]  # Depths of observations

In [ ]:
import torch.nn.functional as F

# Helper which returns smooth L1 loss and NSE
# (as single element tensors)
def compute_loss_and_nse(pred, true):
    non_nan = ~torch.isnan(pred) & ~torch.isnan(true)
    pred = pred[non_nan]
    true = true[non_nan]
    loss = F.smooth_l1_loss(pred, true, reduction='mean')
    modeling_inefficiency = torch.sum((pred - true)**2)/torch.sum((true - torch.mean(true))**2)
    return loss, 1 - modeling_inefficiency  

In [ ]:
# For random param setting [1, n_params], plot true vs predicted at depths
# TODO This depends on example_x, example_y, example_z
def plot_predictions_by_depth(para):
    # Predicted SOC at 20 layers
    pred_y_twenty = fun_model_prediction(para, example_x, args.vertical_mixing)[0, 0:20]

    # Depths of 20 layers (zsoi in code)
    depths_twenty = torch.tensor([1.000000000000000E-002, 4.000000000000000E-002, 9.000000000000000E-002, \
            0.160000000000000, 0.260000000000000, 0.400000000000000, \
            0.580000000000000, 0.800000000000000, 1.06000000000000, \
            1.36000000000000, 1.70000000000000, 2.08000000000000, \
            2.50000000000000, 2.99000000000000, 3.58000000000000, \
            4.27000000000000, 5.06000000000000, 5.95000000000000, \
            6.94000000000000, 8.03000000000000, 9.79500000000000, \
            13.3277669529664, 19.4831291701244, 28.8707244343160, \
            41.9984368640029])[0:20]

    # Predicted SOC at observed depths
    pred_y = fun_model_simu(para, example_x, example_z, args.vertical_mixing)
    loss, nse = compute_loss_and_nse(pred_y, example_y)

    # Plot observations (depth vertically)
    plt.scatter(example_y, example_z, label="Observed")
    plt.scatter(pred_y, example_z, label="Predicted (at observed depths)")
    plt.scatter(pred_y_twenty, depths_twenty, label="Predicted (at default depths)")
    plt.gca().set_ylim(0.2, 2.6)
    plt.gca().invert_yaxis()
    plt.title(f"True vs predicted SOC by depth: NSE={nse.item():.3f}")
    plt.legend()
    plt.show()


In [ ]:
import torch.nn.functional as F

# Try to optimize one site's params
N_INITS = 5
N_STEPS = 500
param_trajectories = torch.zeros((N_INITS, N_STEPS, len(para_names)), requires_grad=False)
loss_trajectories = torch.zeros((N_INITS, N_STEPS), requires_grad=False)
nse_trajectories = torch.zeros((N_INITS, N_STEPS), requires_grad=False)

for INIT_IDX in range(N_INITS):
    params = torch.nn.Parameter(torch.randn([1, len(para_names)]), requires_grad=True)
    optimizer = torch.optim.Adam([params], lr=1e-2)

    for STEP_IDX in range(N_STEPS):
        param_trajectories[INIT_IDX, STEP_IDX, :] = F.sigmoid(params).detach()
        optimizer.zero_grad()
        pred_y = fun_model_simu(F.sigmoid(params), example_x, example_z, args.vertical_mixing)
        loss, nse = compute_loss_and_nse(pred_y, example_y)
        loss.backward()
        optimizer.step()
        loss_trajectories[INIT_IDX, STEP_IDX] = loss.item()
        nse_trajectories[INIT_IDX, STEP_IDX] = nse.item()
        if STEP_IDX % 100 == 99:
            print(f"Init {INIT_IDX} Step {STEP_IDX}: NSE {nse.item()}")

# Plot loss curves
for INIT_IDX in range(N_INITS):
    plt.plot(list(range(N_STEPS)), loss_trajectories[INIT_IDX, :].numpy())
plt.title("Loss curves (optimizing on single example)")
plt.xlabel("Step number")
plt.ylabel("Loss")
plt.show()

# Plot NSE curves
for INIT_IDX in range(N_INITS):
    plt.plot(list(range(N_STEPS)), nse_trajectories[INIT_IDX, :].numpy())
plt.title("NSE curves (optimizing on single example)")
plt.xlabel("Step number")
plt.ylabel("NSE")
plt.ylim(0, 1)
plt.show()


In [ ]:
# Compare to random search
nses = []
for random_idx in range(500):
    # Random param choice
    random_para = torch.rand((1, len(para_names)))
    pred_y = fun_model_simu(random_para, example_x, example_z, args.vertical_mixing)
    loss, nse = compute_loss_and_nse(pred_y, example_y)
    nses.append(nse.item())
print("Best NSE from random search", np.array(nses).max())

In [ ]:
# Check predictions using "optimized parameters"
plot_predictions_by_depth(param_trajectories[3, -1, :].unsqueeze(0).detach())

In [ ]:
# Plot how parameters change in different optimization trajectories
for PARAM_IDX, PARAM_NAME in enumerate(para_names):
    for INIT_IDX in range(N_INITS):
        plt.plot(list(range(N_STEPS)), param_trajectories[INIT_IDX, :, PARAM_IDX].cpu().detach().numpy())
    plt.title(PARAM_NAME + ": optimization trajectory")
    plt.xlabel("Step")
    plt.ylabel(PARAM_NAME)
    plt.show()

In [ ]:
# Check if minima are "connected"
minimum_1 = param_trajectories[0, -1, :].unsqueeze(0)
minimum_2 = param_trajectories[4, -1, :].unsqueeze(0)
print("Local minima 1", minimum_1)
print("Local minima 2", minimum_2)
alphas = np.arange(-0.0, 1.0, 0.1)
nses = []
for alpha in alphas:
    interpolated = minimum_1 * alpha + minimum_2 * (1-alpha)
    pred_y = fun_model_simu(interpolated, example_x, example_z, args.vertical_mixing)
    loss, nse = compute_loss_and_nse(pred_y, example_y)
    non_nan = ~torch.isnan(pred_y) & ~torch.isnan(example_y)
    nses.append(nse.item())

plt.plot(alphas, nses)

In [ ]:
# Starting from local minima, plot how changing single param influences loss
# Loop over parameters
for PARAM_IDX, PARAM_NAME in enumerate(para_names):
    all_nse_curves = []
    
    # Loop over local minima
    for trajectory_idx in range(param_trajectories.shape[0]):
        perturbed_params = param_trajectories[trajectory_idx:trajectory_idx+1, -1, :].clone().detach()
        orig_value = perturbed_params[0, PARAM_IDX].item()
        nses = []

        # Perturb this parameter
        param_values = np.arange(0.05, 1.0, 0.05)
        for param_value in param_values:
            perturbed_params[:, PARAM_IDX] = param_value
            pred_y = fun_model_simu(perturbed_params, example_x, example_z, args.vertical_mixing)
            loss, nse = compute_loss_and_nse(pred_y, example_y)
            nses.append(nse)
        all_nse_curves.append(nses)
        plt.plot(param_values, nses)
        plt.scatter([orig_value], [nse_trajectories[trajectory_idx, -1]])
    plt.title(PARAM_NAME)
    plt.xlabel(PARAM_NAME)
    plt.ylabel("NSE")
    plt.show()


In [ ]:
# Compute 2d representation of params
from sklearn.decomposition import PCA
pca = PCA(n_components=2, svd_solver='full')
trajectories_2d = pca.fit_transform(param_trajectories.reshape(-1, param_trajectories.shape[2]))
trajectories_2d = trajectories_2d.reshape(param_trajectories.shape[0], param_trajectories.shape[1], 2)
print(pca.explained_variance_ratio_)

# Compute a contour
LEVELS = [0, 0.2, 0.4, 0.6, 0.8, 0.85, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99]
min_x, max_x = np.min(trajectories_2d[:, :, 0]), np.max(trajectories_2d[:, :, 0])
min_y, max_y = np.min(trajectories_2d[:, :, 1]), np.max(trajectories_2d[:, :, -1])
x = np.linspace(min_x, max_x, num=20)
y = np.linspace(min_y, max_y, num=20)
X, Y = np.meshgrid(x, y)  # Both have shape [x_vals, y_vals]
Z = np.ones_like(X) * -1
for i in range(X.shape[0]):
    for j in range(Y.shape[1]):
        # Create tensor of shape [1, num_params]
        params = torch.tensor(pca.inverse_transform(np.array([X[i, j], Y[i, j]]))).unsqueeze(0).float()
        if torch.any((params < 0) | (params > 1)):
            Z[i, j] = -1.0
        else:
            pred_y = fun_model_simu(params, example_x, example_z, args.vertical_mixing)
            loss, nse = compute_loss_and_nse(pred_y, example_y)
            Z[i, j] = nse
fig, ax = plt.subplots()
CS = ax.contour(X, Y, Z, levels=LEVELS)
ax.set_title("NSE")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")

# Plot the trajectories in 2d space
for trajectory_idx in range(param_trajectories.shape[0]):
    plt.plot(trajectories_2d[trajectory_idx, :, 0], trajectories_2d[trajectory_idx, :, 1], '-o')

plt.show()

In [ ]:
# Starting from local minima, plot contours as we vary two parameters (and hold others fixed)
LEVELS = [0, 0.2, 0.4, 0.6, 0.8, 0.85, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99]
for trajectory_idx in range(1):  # param_trajectories.shape[0]):
    initial_params = param_trajectories[trajectory_idx:trajectory_idx+1, -1, :].clone().detach()
    print("Initial params", initial_params)

    # Loop over param pairs
    sensitive_params = ["efolding", "tau4s3", "fs1s3", "w-scaling"]
    sensitive_indices = [para_names.index(p) for p in sensitive_params]
    for p1 in range(len(sensitive_params)):
        for p2 in range(p1+1, len(sensitive_params)):
            param1_idx = sensitive_indices[p1]
            param2_idx = sensitive_indices[p2]
            orig_x, orig_y = initial_params[0, param1_idx].item(), initial_params[0, param2_idx].item()
            perturbed_params = initial_params.clone().detach()

            # Create grid of the two params
            x = np.arange(0.05, 0.95, 0.1)  # param1 values
            y = np.arange(0.05, 0.95, 0.1)  # param2 values
            X, Y = np.meshgrid(x, y)  # Both have shape [param1_values, param2_values]

            # Compute NSE at each grid point, store in Z
            Z = np.empty_like(X)
            for i in range(X.shape[0]):
                for j in range(X.shape[1]):
                    perturbed_params[:, param1_idx] = X[i, j]
                    perturbed_params[:, param2_idx] = Y[i, j]
                    pred_y = fun_model_simu(perturbed_params, example_x, example_z, args.vertical_mixing)
                    loss, nse = compute_loss_and_nse(pred_y, example_y)
                    Z[i, j] = nse

            # Plot contour
            fig, ax = plt.subplots()
            ax.scatter(orig_x, orig_y)
            CS = ax.contour(X, Y, Z, levels=LEVELS)
            ax.clabel(CS, inline=True, fontsize=10)
            ax.set_title("NSE")
            ax.set_xlabel(para_names[param1_idx])
            ax.set_ylabel(para_names[param2_idx])
            plt.show()


In [ ]:
# Starting from a local minimum, plot contours as we vary params in two random directions
rng = np.random.default_rng()
LEVELS = [0, 0.2, 0.4, 0.6, 0.8, 0.85, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99]

for trajectory_idx in range(param_trajectories.shape[0]):
    initial_params = torch.logit(param_trajectories[trajectory_idx:trajectory_idx+1, -1, :].clone().detach())  # Pre-sigmoid (logit inverts sigmoid)

    # Compute 2 orthogonal vectors, not sure if this is correct.
    dir1 = torch.rand(size=(len(para_names),)) * 2. - 1
    dir2 = torch.rand(size=(len(para_names),)) * 2. - 1
    dir2 -= torch.dot(dir1, dir2) * dir2

    # Create grid of the two params
    x = np.arange(-2.0, 2.0, 0.05)  # param1 values
    y = np.arange(-2.0, 2.0, 0.05)  # param2 values
    X, Y = np.meshgrid(x, y)  # Both have shape [param1_values, param2_values]

    # Compute NSE at each grid point, store in Z
    Z = np.empty_like(X)
    best_nse = float('-inf')
    best_para = np.nan
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            perturbed_params = F.sigmoid(initial_params.clone().detach() + X[i, j] * dir1 + Y[i, j] * dir2)
            pred_y = fun_model_simu(perturbed_params, example_x, example_z, args.vertical_mixing)
            loss, nse = compute_loss_and_nse(pred_y, example_y)
            Z[i, j] = nse.item()
            if nse > best_nse:
                best_nse = nse
                best_para = perturbed_params

    # Plot contour
    fig, ax = plt.subplots()
    ax.scatter(0, 0)
    CS = ax.contour(X, Y, Z, levels=LEVELS)
    ax.clabel(CS, inline=True, fontsize=10)
    ax.set_title("Loss")
    ax.set_xlabel("Dir1")
    ax.set_ylabel("Dir2")
    plt.show()

    # Plot depthwise prediction of best params
    print("Best para", best_para)
    plot_predictions_by_depth(best_para)

In [ ]:
pca = PCA(n_components=2, svd_solver='full')
trajectories_2d = pca.fit_transform(param_trajectories.reshape(-1, param_trajectories.shape[2]))
print(pca.components_.shape)

In [ ]:
# Starting from a local minimum, plot contours as we vary params in (1) PC1 and (2) random orthogonal dir
rng = np.random.default_rng()
LEVELS = [0, 0.2, 0.4, 0.6, 0.8, 0.85, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99]

for trajectory_idx in range(param_trajectories.shape[0]):
    initial_params = torch.logit(param_trajectories[trajectory_idx:trajectory_idx+1, -1, :].clone().detach())  # Pre-sigmoid (logit inverts sigmoid)

    # Compute 2 orthogonal vectors, not sure if this is correct.
    dir1 = torch.tensor(pca.components_[0, :]).float()
    dir2 = torch.rand(size=(len(para_names),)) * 2. - 1
    dir2 -= torch.dot(dir1, dir2) * dir2

    # Create grid of the two params
    x = np.arange(-2.0, 2.0, 0.05)  # param1 values
    y = np.arange(-2.0, 2.0, 0.05)  # param2 values
    X, Y = np.meshgrid(x, y)  # Both have shape [param1_values, param2_values]

    # Compute NSE at each grid point, store in Z
    Z = np.empty_like(X)
    best_nse = float('-inf')
    best_para = np.nan
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            perturbed_params = F.sigmoid(initial_params.clone().detach() + X[i, j] * dir1 + Y[i, j] * dir2)
            pred_y = fun_model_simu(perturbed_params, example_x, example_z, args.vertical_mixing)
            loss, nse = compute_loss_and_nse(pred_y, example_y)
            Z[i, j] = nse.item()
            if nse > best_nse:
                best_nse = nse
                best_para = perturbed_params

    # Plot contour
    fig, ax = plt.subplots()
    ax.scatter(0, 0)
    CS = ax.contour(X, Y, Z, levels=LEVELS)
    ax.clabel(CS, inline=True, fontsize=10)
    ax.set_title("Loss")
    ax.set_xlabel("Dir1")
    ax.set_ylabel("Dir2")
    plt.show()

    # Plot depthwise prediction of best params
    print("Best para", best_para)
    plot_predictions_by_depth(best_para)

## Analyze trained model's loss landscape